# Homoglyph Attack for Adversarial Spam Texts

This notebook uses Unicode homoglyph substitution to modify spam messages in a subtle but adversarial way. 
Characters are replaced with visually similar Unicode counterparts (e.g., `a` → `а`, `o` → `ο`), which can fool models or filters while looking unchanged to humans.

## Install Dependencies

Install the `homoglyphs` library to perform Unicode character substitutions.

In [2]:
!pip install homoglyphs

     ---------------------------------------- 88.4/88.4 kB 1.0 MB/s eta 0:00:00


## Import Libraries

We import the homoglyph utility, `pandas`, and `random`.

In [2]:
import homoglyphs as hg
import random
import pandas as pd

## Load Dataset

We load the Enron test dataset. Make sure the path matches your project structure.

In [35]:
df = pd.read_csv("../dataset/enron1/enron1_test.csv")  # Adjust path as needed

## Define Homoglyph Obfuscation Function

This function replaces characters with homoglyphs with a given probability. 
Only valid visual substitutes that differ from the original character are used.

In [36]:
# Initialize homoglyph mapper
glyphs = hg.Homoglyphs()

# Character-level homoglyph substitution
def homoglyph_obfuscate(text, replacement_prob=0.25):
    obfuscated = ""
    for char in text:
        # Get a list of homoglyph alternatives for the character
        substitutes = glyphs.get_combinations(char)
        # Filter out the original character so replacements are actually different
        valid_subs = [s for s in substitutes if s != char]
        # With a probability defined by `replacement_prob`, replace the character with a homoglyph
        if valid_subs and random.random() < replacement_prob:
            obfuscated += random.choice(valid_subs)
        else:
            obfuscated += char
    return obfuscated

## Apply Attack to Spam Emails

We apply the homoglyph attack **only** to messages labeled as spam, leaving ham messages untouched.

In [37]:
# Apply homoglyph attack to spam emails only
# Start by duplicating the original email content into a new column
df["email_homoglyph"] = df["email"]
df.loc[df["target"] == "spam", "email_homoglyph"] = df.loc[df["target"] == "spam", "email"].apply(
    lambda x: homoglyph_obfuscate(x, replacement_prob=0.25)
) # Replace characters with homoglyphs with 25% chance

## Preview Obfuscated Spam

Inspect a few rows to see how spam content has been modified with homoglyphs.

In [38]:
df[df["target"] == "spam"]

,email,target,email_homoglyph
2,Subject: re : tittletattle secrets dn ' t ie n...,spam,S𝚞bｊe𝖈t: re : tⅈ𝔱tl𝚎𝓽attle sec𝐫ets dn ′ t ie n...
4,Subject: weekend entertainment alpha male plus...,spam,Su𝓫j𝔢c𝙩: we℮kend ente𝔯𝙩ainmen𝑡 al𝛠ha ma𝟷e plus...
5,Subject: incr ' ease yo ' ur man ' hood by 4 -...,spam,Subje𝒸𝗍: inc𝓻 ʹ ea𝒔e y𝞼 ʹ ur man ' ℎ𝐨od 𝔟y 𝟦 -...
9,"Subject: hiv , hiv , charset = us - ascii "" > ...",spam,"S𝔲bje𝒸t: 𝗵iv ¸ hi𝖛 , charset = us - ａscii "" > ..."
13,"Subject: esplanade davies , govenment don ' t ...",spam,Subject: e𝑠planad𝚎 davie𝖘 ‚ govenm𝗲nt don ' t ...
...,...,...,...
981,Subject: heisser fetish mann war das ein woche...,spam,S𝘶𝙗𝒿ect: heis𝚜𝐞r fet𝗂sh mann wa𝓇 das ｅin w𝝄c𝕙𝚎...
984,"Subject: re : keeping it like a rock hello , d...",spam,"S𝐮b𝑗ect: r𝚎 : keeping it lik𝒆 a roｃk hell𝝾 , d..."
989,Subject: reduce wrinkles h - g - hs most drama...,spam,S𝞾b𝕛ect: reduc𝖊 wrin𝒌𝙡es h - g - hs m𝞂st 𝒹ram𝗮...
991,Subject: chea ; p software looking for extreme...,spam,Su𝙗je𝑐t: c𝘩e𝙖 ; 𝚙 softwaꭇe looking for ℮xtreme...


## Save the Modified Dataset

We export the adversarial version to the homoglyph subfolder for further use.

In [39]:
# Save the modified dataset
df.to_csv("../dataset/enron1/homoglyph/enron1_test_homoglyph.csv", index=False)

In [40]:
df

,email,target,email_homoglyph
0,Subject: unify / sitara enhancements i am comp...,ham,Subject: unify / sitara enhancements i am comp...
1,Subject: weekend activity dated : june 2 thru ...,ham,Subject: weekend activity dated : june 2 thru ...
2,Subject: re : tittletattle secrets dn ' t ie n...,spam,S𝚞bｊe𝖈t: re : tⅈ𝔱tl𝚎𝓽attle sec𝐫ets dn ′ t ie n...
3,"Subject: deal id 109475 daren , the above deal...",ham,"Subject: deal id 109475 daren , the above deal..."
4,Subject: weekend entertainment alpha male plus...,spam,Su𝓫j𝔢c𝙩: we℮kend ente𝔯𝙩ainmen𝑡 al𝛠ha ma𝟷e plus...
...,...,...,...
994,Subject: meter 6387 - dec 00 daren - meter 638...,ham,Subject: meter 6387 - dec 00 daren - meter 638...
995,Subject: re time to reorder v ` icodin looking...,spam,𝑆𝛖bject︰ re time t𝕠 reorder 𝖛 ` icod⍳n lookinᶃ...
996,"Subject: enron actuals for june 21 , 2000 teco...",ham,"Subject: enron actuals for june 21 , 2000 teco..."
997,Subject: re : ena sales on hpl daren - thank y...,ham,Subject: re : ena sales on hpl daren - thank y...
